# A Dose-Response Evaluation of Explanation Metrics under Controlled Shortcut Reliance in CNN Image Classifiers

## Amendment A3, session 3: SD4 watermark pilot and the ROAD timing gate

Pre-registered in `docs/PREDICTIONS_amendments.md` (A3) and `docs/PREDICTIONS_sd3_sd4.md`
(P10–P13), timestamped on Zenodo (10.5281/zenodo.24416017).

- Cell 3: SD4 pilot, ImageNet-pretrained ResNet-18 on Pet with the watermark
  shortcut, doses 0 and +0.25, seed 7 (training only). A3 lets only the pilots'
  reliance and accuracies be read before the dated SD4 completion note.
- Cell 4: ROAD timing on one `pet_resnet18` and one `pet_resnet18_pretrained`
  model (`road --time-only`): only the wall time is kept, the scores are discarded.
- Cell 5: `a3_pilot_report.json` (reliance, accuracies, chance floor, r_c,
  the A3 dose set, the stopping rule, the ROAD times and the 15-minute gate).
- Cell 6: `a3_session3_results.zip` in the notebook's Output.

Before running: **Settings → Accelerator → GPU T4 x2**, and **Internet on**
(Captum and the ImageNet weights). Then **Save Version → Save & Run All (Commit)**.

In [ ]:
# 1. Copy the uploaded project into the writable working folder
import glob, os, shutil, subprocess, sys
found = glob.glob('/kaggle/input/**/dose_response.py', recursive=True)
assert found, 'Project files not found: add your uploaded dataset with "+ Add Input".'
src = os.path.dirname(found[0])
WORK = '/kaggle/working/project'
if not os.path.exists(WORK):
    shutil.copytree(src, WORK)
os.chdir(WORK)
print('project copied from', src)
print(sorted(os.listdir(WORK)))

In [ ]:
# 2. Install the one missing package, check the GPU and the data, get the ImageNet weights
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'captum==0.9.0'], check=True)
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'No GPU: turn on Settings → Accelerator → GPU, then restart.'
for name in ('trainval.npz', 'test.npz'):
    assert os.path.exists(f'data/pet128/{name}'), f'data/pet128/{name} missing: rebuild the bundle with -Dataset a3.'
assert os.path.isdir('data/cifar-10-batches-py'), 'data/cifar-10-batches-py missing: rebuild the bundle with -Dataset a3.'
print('Pet and CIFAR-10 data found')
for name in ('pet_resnet18_patch_dose-050_seed42', 'pet_resnet18_pretrained_patch_dose+000_seed42'):
    for suffix in ('.pt', '_metrics.json'):
        assert os.path.exists(f'checkpoint/dose/{name}{suffix}'), (
            f'checkpoint/dose/{name}{suffix} missing: rebuild the bundle with -Dataset a3.')
print('ROAD timing checkpoints found')
from torchvision.models import ResNet18_Weights, resnet18
try:
    resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)  # cached for the training cells
except Exception as error:
    raise RuntimeError('ImageNet weights download failed: turn Internet on (Settings → Internet).') from error
print('ImageNet ResNet-18 weights ready')

In [ ]:
# 3. SD4 pilot: watermark, doses 0 and +0.25, seed 7 (training only; not scored before the A3 note)
!python -u -W ignore dose_response.py train --bases pet_resnet18_pretrained --shortcuts watermark --doses 0 0.25 --seeds 7

In [ ]:
# 4. ROAD timing gate: one Pet model of each network, scores discarded (road_timing.csv)
!python -u -W ignore dose_response.py road --time-only --models pet_resnet18_patch_dose-050_seed42 pet_resnet18_pretrained_patch_dose+000_seed42

In [ ]:
# 5. Pilot report (a3_pilot_report.json): reliance, accuracies, chance floor, r_c,
#    the A3 dose set (dose-0 pilot), the stopping rule (+0.25 pilot) and the ROAD timing gate
import csv, json
sys.path.insert(0, WORK)
import dose_response
from xai.data import num_classes
from xai.stats import chance_floor, corrected_reliance
BASE = 'pet_resnet18_pretrained'
pilots, rc = [], {}
for dose in (0.0, 0.25):
    name = dose_response.model_name(BASE, 'watermark', dose, 7)
    m = json.load(open(f'checkpoint/dose/{name}_metrics.json'))
    floor = float(chance_floor(m['test_accuracy_no_shortcut'], num_classes(BASE)))
    rc[dose] = float(corrected_reliance(m['reliance'], floor))
    entry = {'model': name, 'reliance': m['reliance'], 'test_accuracy': m['test_accuracy'],
             'test_accuracy_no_shortcut': m['test_accuracy_no_shortcut'],
             'test_accuracy_conflicting': m['test_accuracy_conflicting'], 'chance_floor': round(floor, 2),
             'reliance_corrected': round(rc[dose], 2), 'training_minutes': m['training_minutes']}
    pilots.append(entry)
    print(entry)
with open('results/dose_response/road_timing.csv', newline='') as f:
    timing = [dict(row, minutes=float(row['minutes']), images=int(row['images']),
                   timing_gate_exceeded=float(row['minutes']) > 15) for row in csv.DictReader(f)]
report = {'pilots': pilots,
          'a3_sd4_rule': '60 or above' if rc[0.0] >= 60 else 'below 60',
          'a3_dose_set': dose_response.a3_dose_set(rc[0.0]),
          'sd4_stopped': dose_response.a3_sd4_stopped(rc[0.25]),
          'road_timing': timing}
print({k: v for k, v in report.items() if k != 'pilots'})
json.dump(report, open('a3_pilot_report.json', 'w'), indent=2)

In [ ]:
# 6. Pack the results to download
import zipfile
out = '/kaggle/working/a3_session3_results.zip'
patterns = ['checkpoint/dose/pet_resnet18_pretrained_watermark_*', 'a3_pilot_report.json',
            'results/dose_response/road_timing.csv']
files = sorted({f for p in patterns for f in glob.glob(p)})
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, f)  # keeps checkpoint/... and results/... paths
print(len(files), 'files ->', out)
shutil.rmtree(WORK)  # keep only the zip in the Output